# Actual Payin Extract (`#t7`)

This notebook executes `sql_scripts/jcx_payinDate_harV2.sql` and loads temp table `#t7` into pandas for analysis.

In [3]:
# If needed, install once in this environment:
# %pip install python-dotenv sqlalchemy pyodbc pandas

import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, '..')

from util.db import create_db_engine
from util.sql_loader import read_sql_file
from util.sql_runner import execute_sql_and_read_temp_table

In [4]:
# Paths
project_root = Path.cwd().resolve().parent  # .../yieldCurve_augmenting
env_path = project_root / '.env'
sql_path = project_root / 'sql_scripts' / 'jcx_payinDate_harV2.sql'

engine = create_db_engine(env_path)
setup_sql = read_sql_file(sql_path)


print('SQL loaded from:', sql_path)
print('Using env file:', env_path)

SQL loaded from: /Users/starsrain/2025_concord/yieldCurve_augmenting/sql_scripts/jcx_payinDate_harV2.sql
Using env file: /Users/starsrain/2025_concord/yieldCurve_augmenting/.env


In [3]:
# Execute SQL script and load #t7 into pandas
df_t7 = execute_sql_and_read_temp_table(engine, setup_sql, '#t7')

# Basic typing cleanup for downstream analysis
for col in ['ApplicationDate', 'OriginationDate']:
    if col in df_t7.columns:
        df_t7[col] = pd.to_datetime(df_t7[col], errors='coerce')

print('df_t7 shape:', df_t7.shape)
df_t7.head(10)

df_t7 shape: (2479858, 23)


,LoanID,OriginatedAmount,AppYear,AppMonth,AppWeek,TotalRealizedPayin,InstallmentNumber,PaidOffPaymentAmount,PaymentType,Payment_Number,...,PaymentID,Application_ID,PortFolioID,LoanStatus,NewlyScored,Accepted,Originated,OriginationDate,numOfReturn,numOfPayment
0,I1532072-0,900.0,2023,1,1,3730.0,1,500.0,Installment Pmt,1,...,69521783,104380159,5,D,0,1,1,2023-01-04 15:26:37.283,0.0,11.0
1,I1532072-0,900.0,2023,1,1,3730.0,2,475.0,Installment Pmt,2,...,69521784,104380159,5,D,0,1,1,2023-01-04 15:26:37.283,0.0,11.0
2,I1532072-0,900.0,2023,1,1,3730.0,3,450.0,Installment Pmt,3,...,69521785,104380159,5,D,0,1,1,2023-01-04 15:26:37.283,0.0,11.0
3,I1532072-0,900.0,2023,1,1,3730.0,4,425.0,Installment Pmt,4,...,69521786,104380159,5,D,0,1,1,2023-01-04 15:26:37.283,0.0,11.0
4,I1532072-0,900.0,2023,1,1,3730.0,5,490.0,Installment Pmt,5,...,69521787,104380159,5,D,0,1,1,2023-01-04 15:26:37.283,0.0,11.0
5,I1532072-0,900.0,2023,1,1,3730.0,6,420.0,Installment Pmt,6,...,70520732,104380159,5,D,0,1,1,2023-01-04 15:26:37.283,0.0,11.0
6,I1532072-0,900.0,2023,1,1,3730.0,7,310.0,Installment Pmt,7,...,70520733,104380159,5,D,0,1,1,2023-01-04 15:26:37.283,0.0,11.0
7,I1532072-0,900.0,2023,1,1,3730.0,8,260.0,Installment Pmt,8,...,70520734,104380159,5,D,0,1,1,2023-01-04 15:26:37.283,0.0,11.0
8,I1532072-0,900.0,2023,1,1,3730.0,9,210.0,Installment Pmt,9,...,70520735,104380159,5,D,0,1,1,2023-01-04 15:26:37.283,0.0,11.0
9,I1532072-0,900.0,2023,1,1,3730.0,10,160.0,Installment Pmt,10,...,70520736,104380159,5,D,0,1,1,2023-01-04 15:26:37.283,0.0,11.0


In [4]:
df_t7.columns

Index(['LoanID', 'OriginatedAmount', 'AppYear', 'AppMonth', 'AppWeek',
       'TotalRealizedPayin', 'InstallmentNumber', 'PaidOffPaymentAmount',
       'PaymentType', 'Payment_Number', 'PaymentStatus', 'CustType',
       'Frequency', 'PaymentID', 'Application_ID', 'PortFolioID', 'LoanStatus',
       'NewlyScored', 'Accepted', 'Originated', 'OriginationDate',
       'numOfReturn', 'numOfPayment'],
      dtype='object')

## AppDate Monthly Payin Table (No Installment)

Comprehensive monthly table by `AppDateMonth` and `CustType`, using one `TotalRealizedPayin` value per `LoanID` after filtering out `PaymentStatus == 'R'`.

In [9]:
# Step 1: Non-R cleaning + key field normalization
monthly_base = df_t7.copy()
monthly_base['PaymentStatus'] = monthly_base['PaymentStatus'].astype(str).str.strip()
monthly_base = monthly_base[monthly_base['PaymentStatus'] != 'R'].copy()

for col in ['AppYear', 'AppMonth', 'OriginatedAmount', 'TotalRealizedPayin']:
    monthly_base[col] = pd.to_numeric(monthly_base[col], errors='coerce')

monthly_base['CustType'] = monthly_base['CustType'].astype(str).str.strip()

critical_cols = ['LoanID', 'AppYear', 'AppMonth', 'CustType', 'OriginatedAmount', 'TotalRealizedPayin']
monthly_base = monthly_base.dropna(subset=critical_cols).copy()
monthly_base['AppYear'] = monthly_base['AppYear'].astype(int)
monthly_base['AppMonth'] = monthly_base['AppMonth'].astype(int)
monthly_base['AppDateMonth'] = (
    monthly_base['AppYear'].astype(str)
    + '-'
    + monthly_base['AppMonth'].astype(str).str.zfill(2)
)

# Step 2: Collapse to one row per loan in each monthly cohort
loan_level_monthly = (
    monthly_base.groupby(['LoanID', 'AppYear', 'AppMonth', 'AppDateMonth', 'CustType'], as_index=False)
    .agg(
        OriginatedAmount=('OriginatedAmount', 'max'),
        TotalRealizedPayin=('TotalRealizedPayin', 'max'),
    )
)

print('Monthly base rows (non-R):', len(monthly_base))
print('Loan-level monthly rows:', len(loan_level_monthly))
loan_level_monthly.head(10)

Monthly base rows (non-R): 1776505
Loan-level monthly rows: 188105


,LoanID,AppYear,AppMonth,AppDateMonth,CustType,OriginatedAmount,TotalRealizedPayin
0,I1529359-0,2023,1,2023-01,RETURN,1500.0,2109.68
1,I1529362-0,2023,1,2023-01,RETURN,600.0,763.57
2,I1529363-0,2023,1,2023-01,RETURN,300.0,358.50
3,I1529364-0,2023,1,2023-01,RETURN,500.0,700.00
4,I1529365-0,2023,1,2023-01,RETURN,1500.0,2461.00
5,I1529366-0,2023,1,2023-01,RETURN,1500.0,1800.00
6,I1529367-0,2023,1,2023-01,RETURN,1500.0,4130.00
7,I1529369-0,2023,1,2023-01,NEW,1000.0,2157.16
8,I1529371-0,2023,1,2023-01,NEW,800.0,1696.00
9,I1529372-0,2023,1,2023-01,RETURN,500.0,1050.00


In [10]:
# Step 3: Aggregate to AppDateMonth + CustType
appdate_monthly_payin = (
    loan_level_monthly.groupby(['AppDateMonth', 'CustType'], as_index=False)
    .agg(
        LoanCnt=('LoanID', 'nunique'),
        TotalRealizedPayin=('TotalRealizedPayin', 'sum'),
        OriginatedAmount=('OriginatedAmount', 'sum'),
    )
)
appdate_monthly_payin['PayinRatio'] = (
    appdate_monthly_payin['TotalRealizedPayin'] / appdate_monthly_payin['OriginatedAmount']
)

appdate_monthly_payin = appdate_monthly_payin.sort_values(['AppDateMonth', 'CustType']).reset_index(drop=True)
appdate_monthly_payin.head(20)

,AppDateMonth,CustType,LoanCnt,TotalRealizedPayin,OriginatedAmount,PayinRatio
0,2023-01,NEW,2179,2997723.63,1591150.0,1.883998
1,2023-01,RETURN,2564,3777390.83,2120500.0,1.781368
2,2023-02,NEW,2243,2702376.93,1613400.0,1.674958
3,2023-02,RETURN,2442,3551244.47,2079250.0,1.707945
4,2023-03,NEW,2687,3279441.25,1947800.0,1.683664
5,2023-03,RETURN,2930,4249701.75,2479850.0,1.713693
6,2023-04,NEW,2937,3415730.35,2138850.0,1.596994
7,2023-04,RETURN,2726,4026116.83,2325900.0,1.730993
8,2023-05,NEW,3081,3627573.61,2230950.0,1.626022
9,2023-05,RETURN,3181,4697913.55,2703450.0,1.737748


In [12]:
output_path_v2 = "/Users/starsrain/2025_concord/yieldCurve_augmenting/payin_out/cohort_payin_all_v1.xlsx"
appdate_monthly_payin.to_excel(output_path_v2, index=False)
print(f"Excel file exported to: {output_path_v2}")

Excel file exported to: /Users/starsrain/2025_concord/yieldCurve_augmenting/payin_out/cohort_payin_all_v1.xlsx


# PaymentDate Monthly Cohort Payin (`#t8`)

Monthly realized payin view by `PayYear`, `PayMonth`, and `CustType`, using one `TotalRealizedPayin` value per `LoanID` after filtering out `PaymentStatus == 'R'`.

In [9]:
# Step 1: Execute SQL and read #t8 explicitly (same connection/session)

# Be robust if this cell is run independently before setup_sql is defined.
if 'setup_sql' not in globals():
    setup_sql = read_sql_file(sql_path)

with engine.connect() as conn:
    conn.exec_driver_sql(setup_sql)
    df_t8 = pd.read_sql_query('SELECT * FROM #t8', conn)


df_t8.head(10)

,LoanID,OriginatedAmount,PaymentDate,TransactionDate,TxYear,TxMonth,TxWeek,AppYear,AppMonth,AppWeek,...,PaymentID,Application_ID,PortFolioID,LoanStatus,NewlyScored,Accepted,Originated,OriginationDate,numOfReturn,numOfPayment
0,I2202630-0,800.0,2024-12-18,2024-12-17 15:13:26.713,2024.0,12.0,51.0,2024,11,48,...,73547542,131735809,5,C,0,1,1,2024-11-27 10:22:34.243,16,19
1,I2202630-0,800.0,2024-12-19,2024-12-19 14:25:28.737,2024.0,12.0,51.0,2024,11,48,...,73561329,131735809,5,C,0,1,1,2024-11-27 10:22:34.243,16,19
2,I2202630-0,800.0,2024-12-20,2024-12-20 06:15:12.230,2024.0,12.0,51.0,2024,11,48,...,73561653,131735809,5,C,0,1,1,2024-11-27 10:22:34.243,16,19
3,I2202630-0,800.0,2024-12-27,2024-12-27 06:14:32.467,2024.0,12.0,52.0,2024,11,48,...,73569429,131735809,5,C,0,1,1,2024-11-27 10:22:34.243,16,19
4,I2202630-0,800.0,2025-01-10,2025-01-10 06:14:48.113,2025.0,1.0,2.0,2024,11,48,...,73653851,131735809,5,C,0,1,1,2024-11-27 10:22:34.243,16,19
5,I2202630-0,800.0,2025-01-17,2025-01-17 06:16:07.757,2025.0,1.0,3.0,2024,11,48,...,73653852,131735809,5,C,0,1,1,2024-11-27 10:22:34.243,16,19
6,I2202630-0,800.0,2025-01-21,2025-01-17 11:23:01.747,2025.0,1.0,3.0,2024,11,48,...,73687001,131735809,5,C,0,1,1,2024-11-27 10:22:34.243,16,19
7,I2202630-0,800.0,2025-01-22,2025-01-22 15:26:00.063,2025.0,1.0,4.0,2024,11,48,...,73709274,131735809,5,C,0,1,1,2024-11-27 10:22:34.243,16,19
8,I2202630-0,800.0,2025-01-24,2025-01-24 05:56:21.470,2025.0,1.0,4.0,2024,11,48,...,73710610,131735809,5,C,0,1,1,2024-11-27 10:22:34.243,16,19
9,I2202630-0,800.0,2025-01-27,2025-01-27 05:13:51.317,2025.0,1.0,5.0,2024,11,48,...,73717595,131735809,5,C,0,1,1,2024-11-27 10:22:34.243,16,19


In [10]:
df_t8.columns

Index(['LoanID', 'OriginatedAmount', 'PaymentDate', 'TransactionDate',
       'TxYear', 'TxMonth', 'TxWeek', 'AppYear', 'AppMonth', 'AppWeek',
       'TotalRealizedPayin', 'InstallmentNumber', 'PaidOffPaymentAmount',
       'PaymentType', 'Payment_Number', 'PaymentStatus', 'CustType',
       'Frequency', 'PaymentID', 'Application_ID', 'PortFolioID', 'LoanStatus',
       'NewlyScored', 'Accepted', 'Originated', 'OriginationDate',
       'numOfReturn', 'numOfPayment'],
      dtype='object')

In [11]:
# TransactionDate-anchored cohort payin using PaidOffPaymentAmount
for col in ['PaymentDate', 'TransactionDate', 'OriginationDate']:
    if col in df_t8.columns:
        df_t8[col] = pd.to_datetime(df_t8[col], errors='coerce')

# Use TransactionDate as requested
df_t8['TxYear'] = df_t8['TransactionDate'].dt.year
df_t8['TxMonth'] = df_t8['TransactionDate'].dt.month

base_tx = df_t8.copy()
base_tx['PaymentStatus'] = base_tx['PaymentStatus'].astype(str).str.strip()
base_tx = base_tx[base_tx['PaymentStatus'] != 'R'].copy()

for col in ['TxYear', 'TxMonth', 'PaidOffPaymentAmount', 'OriginatedAmount']:
    base_tx[col] = pd.to_numeric(base_tx[col], errors='coerce')
base_tx['CustType'] = base_tx['CustType'].astype(str).str.strip()

base_tx = base_tx.dropna(subset=['LoanID', 'TxYear', 'TxMonth', 'CustType', 'PaidOffPaymentAmount'])
base_tx['TxYear'] = base_tx['TxYear'].astype(int)
base_tx['TxMonth'] = base_tx['TxMonth'].astype(int)

# Loan-level payin within each Tx cohort: sum payment amounts per loan
loan_level_tx_paid = (
    base_tx.groupby(['LoanID', 'TxYear', 'TxMonth', 'CustType'], as_index=False)
    .agg(
        LoanPaidOffPaymentAmount=('PaidOffPaymentAmount', 'sum'),
        OriginatedAmount=('OriginatedAmount', 'max'),
    )
)

# Cohort-level payin: sum loan-level payment amounts
tx_monthly_paidoff_payin = (
    loan_level_tx_paid.groupby(['TxYear', 'TxMonth', 'CustType'], as_index=False)
    .agg(
        LoanCnt=('LoanID', 'nunique'),
        TotalPayin_PaidOffPaymentAmount=('LoanPaidOffPaymentAmount', 'sum'),
        OriginatedAmount=('OriginatedAmount', 'sum'),
    )
)
tx_monthly_paidoff_payin['PayinRatio'] = (
    tx_monthly_paidoff_payin['TotalPayin_PaidOffPaymentAmount'] / tx_monthly_paidoff_payin['OriginatedAmount']
)

tx_monthly_paidoff_payin = tx_monthly_paidoff_payin.sort_values(['TxYear', 'TxMonth', 'CustType']).reset_index(drop=True)
tx_monthly_paidoff_payin.head(20)

,TxYear,TxMonth,CustType,LoanCnt,TotalPayin_PaidOffPaymentAmount,OriginatedAmount,PayinRatio
0,2023,1,NEW,1036,356919.58,767300.0,0.465163
1,2023,1,RETURN,1178,607153.05,993600.0,0.611064
2,2023,2,NEW,2819,1161987.74,2035250.0,0.570931
3,2023,2,RETURN,3041,1804430.41,2570000.0,0.702113
4,2023,3,NEW,4468,2080267.41,3200800.0,0.649921
5,2023,3,RETURN,4560,2912029.33,3914050.0,0.743994
6,2023,4,NEW,5240,2135864.19,3795800.0,0.562691
7,2023,4,RETURN,4847,2789906.76,4211950.0,0.662379
8,2023,5,NEW,6419,2637302.00,4616300.0,0.571302
9,2023,5,RETURN,5953,3365737.98,5103900.0,0.659444


In [12]:

output_path_v3 = "/Users/starsrain/2025_concord/yieldCurve_augmenting/payin_out/cohort_payin_transactionDate_paidoff_v2.xlsx"
tx_monthly_paidoff_payin.to_excel(output_path_v3, index=False)
print(f"Excel file exported to: {output_path_v3}")

Excel file exported to: /Users/starsrain/2025_concord/yieldCurve_augmenting/payin_out/cohort_payin_transactionDate_paidoff_v2.xlsx
